# Host Prophage Coverage And Activity

PropagAtE-style checks use its default activity thresholds: ratio >=2, Cohen's d >=0.70, mean prophage depth >=1x and breadth >=50% at >=1x. Scaffold ends are masked by 150 bp, and prophages must be >=1 kb. All QC-passed paired and orphan reads are mapped to the assigned unmasked host; residual host/prophage stages are not used. Secondary and supplementary alignments do not add depth.

MOSAIC's agreed mapping filters remain >=95% identity and >=85% aligned read length, rather than PropagAtE's default 97% identity. This is not an identical stock PropagAtE run. Activity calls describe coverage enrichment, not confirmed induction in a phage-enriched isolate library. Zero host background, short prophages and whole-contig viral candidates remain explicitly not assessed. This report never changes contig retention.

Method: [PropagAtE documentation](https://github.com/AnantharamanLab/PropagAtE).


In [ ]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO
from IPython.display import display, Image, Markdown

sns.set_style("ticks", {"axes.grid": True})
sns.set_palette("colorblind")
plt.rcParams.update({"axes.linewidth": 1.5, "axes.labelsize": 14, "axes.titlesize": 14,
                     "xtick.labelsize": 10, "ytick.labelsize": 10, "svg.fonttype": "none",
                     "font.family": "sans-serif", "font.sans-serif": ["Liberation Sans", "DejaVu Sans"]})
regions = pd.read_csv(snakemake.input.regions, sep="\t")
assignments = pd.read_csv(snakemake.input.assignments, sep="\t").set_index("sample")["expected_host"].to_dict()
host_lengths = {}
for host, path in zip(snakemake.params.hosts, snakemake.input.host_fastas):
    with open(path) as handle:
        host_lengths[host] = {record.id: len(record) for record in SeqIO.parse(handle, "fasta")}
figure_dir = Path(snakemake.output.figures)
figure_dir.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame([dict(min_ratio=snakemake.params.min_ratio,
                          min_cohen_d=snakemake.params.min_cohen_d,
                          min_mean_depth=snakemake.params.min_mean_depth,
                          min_breadth_percent=snakemake.params.min_breadth_percent,
                          scaffold_end_mask_bp=snakemake.params.mask_bp,
                          min_prophage_length_bp=snakemake.params.min_length_bp)]))


## Per-Base Coverage Checks

The background is the same host scaffold outside the union of all its embedded prophage intervals. Unreported zero-depth positions are restored. Masked scaffold ends are excluded from means, variances and breadth denominators; the assessed length is reported. Cohen's d uses the unweighted pooled variance used by PropagAtE, not a length-weighted pooled variance. No smoothing or binning affects the calls.

Active: all four activity thresholds pass. Ambiguous: ratio and effect pass, but mean depth or breadth does not. Dormant: no activity enrichment passes. These are coverage-based estimates; a dormant call is not proof of inactivity. No prophage coverage is not_present. No measurable host background is not_assessed_host_background.


In [ ]:
def prophage_activity_metrics(depth, start, end, embedded_intervals, evidence_scope, params):
    # Coordinates in the evidence table are 1-based and inclusive.
    start, end = min(start, end) - 1, max(start, end)
    valid = np.ones(len(depth), dtype=bool)
    if params.mask_bp:
        valid[:params.mask_bp] = False
        valid[-params.mask_bp:] = False
    background = valid.copy()
    for left, right in embedded_intervals:
        background[min(left, right) - 1:max(left, right)] = False
    if evidence_scope != "embedded_prophage":
        background[:] = False
    phage = depth[start:end][valid[start:end]]
    host = depth[background]
    phage_mean = float(phage.mean()) if phage.size else np.nan
    host_mean = float(host.mean()) if host.size else np.nan
    phage_sd = float(phage.std()) if phage.size else np.nan
    host_sd = float(host.std()) if host.size else np.nan
    breadth = 100 * np.count_nonzero(phage >= params.min_mean_depth) / phage.size if phage.size else np.nan
    ratio = phage_mean / host_mean if host_mean > 0 else np.nan
    pool = np.sqrt((phage_sd ** 2 + host_sd ** 2) / 2)
    effect = abs(phage_mean - host_mean) / pool if pool > 0 else (
        np.inf if np.isfinite(phage_mean) and np.isfinite(host_mean) and phage_mean != host_mean else 0.0)
    passes_ratio = bool(ratio >= params.min_ratio)
    passes_effect = bool(effect >= params.min_cohen_d)
    passes_depth = bool(phage_mean >= params.min_mean_depth)
    passes_breadth = bool(breadth >= params.min_breadth_percent)
    if evidence_scope != "embedded_prophage":
        status = "not_assessed_whole_contig"
    elif end - start < params.min_length_bp:
        status = "not_assessed_short_prophage"
    elif not phage.size:
        status = "not_assessed_after_end_mask"
    elif phage_mean == 0:
        status = "not_present"
    elif not host.size or not host_mean > 0:
        status = "not_assessed_host_background"
    elif passes_ratio and passes_effect:
        status = "active" if passes_depth and passes_breadth else "ambiguous"
    else:
        status = "dormant"
    return dict(prophage_mean_depth=phage_mean, prophage_median_depth=float(np.median(phage)) if phage.size else np.nan,
                prophage_sd_depth=phage_sd, prophage_breadth_percent=breadth,
                prophage_covered_bases=int(np.count_nonzero(phage > 0)),
                assessed_prophage_length_bp=int(phage.size), host_background_length_bp=int(host.size),
                host_mean_depth=host_mean, host_sd_depth=host_sd,
                host_breadth_percent=100 * np.count_nonzero(host >= 1) / host.size if host.size else np.nan,
                prophage_host_ratio=ratio, cohen_d=effect,
                passes_ratio=passes_ratio, passes_cohen_d=passes_effect,
                passes_mean_depth=passes_depth, passes_breadth=passes_breadth,
                activity=status, mapping_identity_percent=95, mapping_aligned_percent=85)


In [ ]:
rows, profiles = [], []
for sample, path in zip(snakemake.params.samples, snakemake.input.depth):
    host = assignments[sample]
    selected = regions[regions["host"].eq(host)]
    depths = {parent: np.zeros(host_lengths[host][parent], dtype=float)
              for parent in selected["parent_contig"].unique()}
    with gzip.open(path, "rt") as handle:
        has_coverage = bool(handle.read(1))
    if has_coverage:
        for chunk in pd.read_csv(path, sep="\t", names=["contig", "position", "depth"], chunksize=100000):
            for parent, values in chunk[chunk["contig"].isin(depths)].groupby("contig"):
                depths[parent][values["position"].to_numpy(dtype=int)] = values["depth"].to_numpy(dtype=float)
    for region in selected.itertuples(index=False):
        depth = depths[region.parent_contig]
        embedded = selected[selected["parent_contig"].eq(region.parent_contig) & selected["evidence_scope"].eq("embedded_prophage")]
        intervals = list(zip(embedded["start"].astype(int), embedded["end"].astype(int)))
        metrics = prophage_activity_metrics(depth, int(region.start), int(region.end), intervals,
                                           region.evidence_scope, snakemake.params)
        rows.append(dict(sample=sample, **region._asdict(), **metrics))
        if region.evidence_scope == "embedded_prophage" and metrics["prophage_mean_depth"] >= snakemake.params.min_mean_depth:
            left = max(0, int(region.start) - 1 - 5000)
            right = min(len(depth), int(region.end) + 5000)
            edges = np.unique(np.linspace(left, right, min(2001, right - left + 1), dtype=int))
            profiles.append(dict(sample=sample, viral_id=region.viral_id, host=host,
                                 start=int(region.start), end=int(region.end), activity=metrics["activity"],
                                 ratio=metrics["prophage_host_ratio"], host_depth=metrics["host_mean_depth"],
                                 positions=(edges[:-1] + edges[1:]) / 2,
                                 depth=np.array([depth[a:b].mean() for a, b in zip(edges[:-1], edges[1:])])))
activity = pd.DataFrame(rows)
if activity.empty:
    activity = pd.DataFrame(columns=["sample"] + regions.columns.tolist() + list(prophage_activity_metrics(
        np.zeros(2000), 1, 1000, [(1, 1000)], "embedded_prophage", snakemake.params)))
activity.to_csv(snakemake.output.table, sep="\t", index=False)
display(activity)
display(activity.groupby(["host", "activity"]).size().unstack(fill_value=0))


## Coverage Enrichment Across Samples

Only regions with some prophage coverage are plotted. Colour is log2(prophage/host mean depth): 0 means equal depth and 1 means two-fold enrichment. An asterisk marks active by the activity thresholds; a question mark marks ambiguous. Blank cells can be undetected, not assessed or from a different assigned host; consult the table. The activity flag is not proof of induction.


In [ ]:
present = activity[activity["prophage_mean_depth"].gt(0) & activity["evidence_scope"].eq("embedded_prophage")]
identifiers = present["viral_id"].drop_duplicates().tolist()
matrix = activity[activity["viral_id"].isin(identifiers)].pivot(index="viral_id", columns="sample", values="prophage_host_ratio").reindex(columns=snakemake.params.samples)
classes = activity[activity["viral_id"].isin(identifiers)].pivot(index="viral_id", columns="sample", values="activity").reindex(index=matrix.index, columns=matrix.columns)
stars = classes.replace({"active": "*", "ambiguous": "?"}).where(classes.isin(["active", "ambiguous"]), "")
fig, ax = plt.subplots(figsize=(max(8, 0.25 * len(matrix.columns) + 3), max(4, 0.3 * len(matrix) + 2)))
if matrix.empty:
    ax.text(0.5, 0.5, "No embedded prophage coverage", ha="center", va="center")
    ax.set_axis_off()
else:
    sns.heatmap(np.log2(matrix.replace(0, np.nan)), cmap="vlag", center=0, vmin=-2, vmax=4,
                annot=stars, fmt="", ax=ax, xticklabels=True, yticklabels=True,
                cbar_kws={"label": "log2(prophage / host mean depth)"})
    ax.set_xlabel("Sample: assigned-host comparison")
    ax.set_ylabel("Host prophage")
    ax.set_title("* activity thresholds pass; ? insufficient depth/breadth support")
fig.savefig(snakemake.output.png, dpi=300, bbox_inches="tight")
fig.savefig(snakemake.output.svg, bbox_inches="tight")
plt.close(fig)
display(Image(filename=snakemake.output.png))


## Prophage Coverage Profiles

Profiles are shown for embedded prophages with mean depth at least the configured minimum. Shading shows the predicted prophage interval; the dashed line is the mean host background from the same scaffold, excluding all embedded prophages. Binning is for display only. Every region, including those without a displayed profile, remains in the results table.


In [ ]:
profile_index = []
for profile in profiles:
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(profile["positions"], profile["depth"], color="#0173b2", linewidth=1)
    ax.axvspan(profile["start"] - 1, profile["end"], color="#de8f05", alpha=0.2, label="Predicted prophage")
    ax.axhline(profile["host_depth"], color="grey", linestyle="--", label="Mean host background")
    ratio_text = f"{profile['ratio']:.2f}" if np.isfinite(profile["ratio"]) else "not assessed"
    ax.set_title(f"{profile['sample']} | {profile['viral_id']}\n{profile['activity']}; prophage/host ratio: {ratio_text}", fontsize=11)
    ax.set_xlabel("Position on host scaffold (bp)")
    ax.set_ylabel("Read depth")
    ax.legend(frameon=False)
    png = figure_dir / (profile["sample"] + "__" + profile["viral_id"] + ".png")
    svg = png.with_suffix(".svg")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)
    profile_index.append(dict(sample=profile["sample"], viral_id=profile["viral_id"], png=str(png), svg=str(svg)))
    display(Image(filename=str(png)))
pd.DataFrame(profile_index, columns=["sample", "viral_id", "png", "svg"]).to_csv(
    figure_dir / "coverage_profiles.tsv", sep="\t", index=False)
